In [7]:
import numpy as np
import matplotlib.pyplot as plt
from upkit import Fit, Histo, tools
import pandas as pd
from lund_io import read_lund
from kinematics import compute_kinematics, compute_kinematics_batch
import vector as vec
%matplotlib qt

mass_p = 0.9382720813

In [2]:
out = compute_kinematics_batch('../events_rescattering.lund')

In [4]:
p_p1 = out['p1']
p_p2 = out['p2']
p_pbar = out['p_miss']
p_e = out['e']
p_beam = out['p_beam']
p_target = out['p_target']

In [29]:
def scramble_protons(p1, p2):
    mask = np.random.rand(len(p1)) < 0.5
    p1_scrambled = vec.array({
        "px": np.where(mask, p1.px, p2.px),
        "py": np.where(mask, p1.py, p2.py),
        "pz": np.where(mask, p1.pz, p2.pz),
        "M": np.where(mask, p1.M, p2.M),
    })
    p2_scrambled = vec.array({
        "px": np.where(mask, p2.px, p1.px),
        "py": np.where(mask, p2.py, p1.py),
        "pz": np.where(mask, p2.pz, p1.pz),
        "M": np.where(mask, p2.M, p1.M),
    })
    return p1_scrambled, p2_scrambled

p_p1, p_p2 = scramble_protons(p_p1, p_p2)

In [30]:
p_p1pbar = p_p1 + p_pbar
p_p2pbar = p_p2 + p_pbar
p_p1p2 = p_p1 + p_p2

p_q = p_beam - p_e

In [31]:
fig, ax = plt.subplots()
h_p1pbar = Histo(p_p1pbar.mass, bins=100, range=(2*mass_p, 3.5), density=True, ax=ax, label='p1 + pbar', alpha=0.5)
h_p2pbar = Histo(p_p2pbar.mass, bins=100, range=(2*mass_p, 3.5), density=True, ax=ax, label='p2 + pbar', alpha=0.5)
h_p1p2 = Histo(p_p1p2.mass, bins=100, range=(2*mass_p, 3.5), density=True, ax=ax, label='p1 + p2', alpha=0.5)

In [32]:
t_qp1pbar = -(p_q - p_p1pbar).M2
t_qp2pbar = -(p_q - p_p2pbar).M2
t_qp1p2 = -(p_q - p_p1p2).M2

In [40]:
fig, ax = plt.subplots()
h_tp1pbar = Histo(t_qp1pbar, bins=100, range=(0, 15), ax=ax, label='p1 + pbar', alpha=0.5)
h_tp2pbar = Histo(t_qp2pbar, bins=100, range=(0, 15), ax=ax, label='p2 + pbar', alpha=0.5)
h_tp1p2 = Histo(t_qp1p2, bins=100, range=(0, 15), ax=ax, label='p1 + p2', alpha=0.5)

In [41]:
h_t_sum = h_tp1pbar + h_tp2pbar
plt.close()
h_t_sum.plot(bins=100, range=(0, 15), label='p1 + pbar + p2 + pbar', alpha=0.5)

h_t_X = h_t_sum - h_tp1p2
plt.close()

h_t_X.plot(bins=100, range=(0, 15), label='p1 + pbar + p2 + pbar - p1 + p2', alpha=0.5)

In [50]:
s_rp = p_p1p2.M2
s_rpbar = p_p1pbar.M2
s_rp2pbar = p_p2pbar.M2

In [51]:
Histo(s_rp, bins=100, range=(4*mass_p**2, 12), label='p1 + p2', alpha=0.5)
Histo(s_rpbar, bins=100, range=(4*mass_p**2, 12), label='p1 + pbar', alpha=0.5)
Histo(s_rp2pbar, bins=100, range=(4*mass_p**2, 12), label='p2 + pbar', alpha=0.5)

In [70]:
import numpy as np
M = 0.938272
def read_mix(path):
    cls, rec, pro, pb = [], [], [], []
    with open(path) as f:
        for line in f:
            h = line.split(); n = int(h[0]); c = int(float(h[8]))
            P = {}
            for _ in range(n):
                r = next(f).split()
                P.setdefault(int(r[3]), []).append([float(r[9]), float(r[6]), float(r[7]), float(r[8])])
            cls.append(c); rec.append(P[2212][0]); pro.append(P[2212][1]); pb.append(P[-2212][0])
    return np.array(cls), np.array(rec), np.array(pro), np.array(pb)

cls, rec, pro, pb = read_mix("../events_rescattering_fsi.lund")
m2 = lambda v: v[:, 0]**2 - (v[:, 1:]**2).sum(1)
t     = m2(np.array([M, 0, 0, 0]) - rec)
s_rpb = m2(rec + pb)
s_pp  = m2(rec + pro)

s_edges = np.quantile(s_rpb[cls == 1], [0, .25, .5, .75, 1])
t_edges = np.linspace(-1.5, 0, 7)
R, dR = [], []
for lo, hi in zip(s_edges[:-1], s_edges[1:]):
    a, _ = np.histogram(t[(cls == 1) & (s_rpb >= lo) & (s_rpb < hi)], t_edges)
    b, _ = np.histogram(t[(cls == 2) & (s_pp  >= lo) & (s_pp  < hi)], t_edges)
    R.append(a / b); dR.append(a / b * np.sqrt(1/a + 1/b))

In [71]:
# ---- Rescattering ratio from counts: binning ----
# Builds on cls, t, s_rpb, s_pp from the read_mix cell above.
# cls is the LUND header's 9th column: 1 = pbar-p run, 2 = p-p run.
# Input models, used only for the overlay: the ones in input.txt. Swap in the real ones.
# sigma_pbarp = lambda s, t: np.exp(4.0*t + 3)
# sigma_pp    = lambda s, t: np.exp(2.0*t + 1)

# # Slopes run with ln(s/s0), s0 = 3.52 GeV^2 (threshold): B(s) = B0 + 2 alpha' ln(s/s0).
sigma_pbarp = lambda s, t: np.exp(3.0 + (4.0 - 1.0*np.log(s/3.52))*t)   # alpha' = -0.5
sigma_pp    = lambda s, t: np.exp(1.0 + (2.0 + 0.5*np.log(s/3.52))*t)   # alpha' = +0.25

is_pbarp, is_pp = cls == 1, cls == 2
s_edges      = np.quantile(s_rpb[is_pbarp], [0, .25, .5, .75, 1])  # 4 s bins, equal pbar-p stats
t_edges      = np.linspace(-1.5, 0, 11)   # ratio
t_edges_dist = np.linspace(-3.0, 0, 31)   # individual spectra
s_bins = list(zip(s_edges[:-1], s_edges[1:]))

def in_s(s, lo, hi):
    return (s >= lo) & (s < hi)

# Palette: reference categorical slots 1-2 (blue, orange) and text/grid tokens.
C_PBARP, C_PP = "#2a78d6", "#eb6834"
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"

def style(ax, left=True):
    ax.grid(True, color=GRID, lw=0.8); ax.set_axisbelow(True)
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    for sp in ("left", "bottom"): ax.spines[sp].set_color(AXIS)
    ax.tick_params(colors=INK2, labelsize=8, labelleft=left)

In [72]:
# ---- t spectra with the ratio underneath: with labels vs without ----
# Uses cell A (sigma_pbarp, sigma_pp, is_pbarp, is_pp, s_bins, t_edges, in_s, colours, style).
# With labels:    pbar-p events binned in s_rpb  over  p-p events binned in s_pp.
# Without labels: ALL events binned in s_rpb     over  ALL events binned in s_pp.
#                 cls is never used, and one event can land in both histograms.
t_c, t_w = 0.5*(t_edges[1:] + t_edges[:-1]), np.diff(t_edges)
all_events = np.ones_like(t, dtype=bool)

def ratio_hists(num, den):
    """The two t histograms, their ratio and its error. Events in BOTH histograms
    (only possible without labels) are correlated: Var(ln R) = 1/a + 1/b - 2z/(ab)."""
    a, _ = np.histogram(t[num], t_edges)
    b, _ = np.histogram(t[den], t_edges)
    z, _ = np.histogram(t[num & den], t_edges)
    ok = (a > 0) & (b > 0)
    with np.errstate(divide="ignore", invalid="ignore"):
        r  = np.where(ok, a / b, np.nan)
        dr = r * np.sqrt(1/a + 1/b - 2*z/(a*b))
    return a, b, r, dr, ok

cases = [
    (r"With labels: $\bar pp$ events binned in $s_{r\bar p}$ over $pp$ events binned in $s_{pp}$",
     r"$\bar pp$ events, in $s_{r\bar p}$", r"$pp$ events, in $s_{pp}$", is_pbarp, is_pp),
    (r"Without labels: all events binned in $s_{r\bar p}$ over all events binned in $s_{pp}$",
     r"all events, in $s_{r\bar p}$", r"all events, in $s_{pp}$", all_events, all_events),
]
tt = np.linspace(t_edges[0], t_edges[-1], 200)
fig = plt.figure(figsize=(14, 10.5), layout="constrained")
dist_rows = []
for sub, (name, lab_num, lab_den, sel_num, sel_den) in zip(fig.subfigures(2, 1, hspace=0.04), cases):
    axes = sub.subplots(2, len(s_bins), sharex=True, sharey="row",
                        gridspec_kw=dict(height_ratios=[3, 1.4]))
    dist_rows.append(axes[0, 0])
    for j, (lo, hi) in enumerate(s_bins):
        num = sel_num & in_s(s_rpb, lo, hi)
        den = sel_den & in_s(s_pp,  lo, hi)
        a, b, r, dr, ok = ratio_hists(num, den)

        # Top: the two spectra whose ratio is plotted below.
        ax = axes[0, j]
        # Numerator drawn larger, so it still shows where the two coincide (no labels).
        for n, color, ms, label in ((a, C_PBARP, 4, lab_num), (b, C_PP, 4, lab_den)):
            m = n > 0
            ax.errorbar(-t_c[m], n[m]/t_w[m], yerr=np.sqrt(n[m])/t_w[m], fmt=".", ms=ms,
                        color=color, capsize=0, label=label)
        ax.set_yscale("log")
        ax.set_title(fr"${lo:.2f} < s < {hi:.2f}$ GeV$^2$", fontsize=10, color=INK)

        # Bottom: the ratio. Reference = the input averaged over the denominator's own
        # events in each bin (with labels), or 1 (without labels: t-only models are degenerate).
        k = np.digitize(t[den], t_edges) - 1
        t_mean = np.array([t[den][k == i].mean() if ok[i] else np.nan for i in range(len(a))])
        key = is_pp & in_s(s_pp, lo, hi)
        k_key = np.digitize(t[key], t_edges) - 1
        answer = np.array([np.mean(sigma_pbarp(s_pp[key][k_key == i], t[key][k_key == i]) /
                                   sigma_pp(s_pp[key][k_key == i], t[key][k_key == i]))
                           if (k_key == i).any() else np.nan for i in range(len(a))])
        ref, ref_name = (answer, "input") if sel_num is is_pbarp else (np.ones(len(a)), "1")
        pull = (r - ref) / dr
        rx = axes[1, j]
        rx.axhline(1, color=AXIS, lw=1)
        rx.stairs(answer, -t_edges, color=MUTED, lw=2, ls="--", baseline=None,
                  label=r"input $\sigma_{\bar pp}/\sigma_{pp}$, bin-averaged", zorder=1)
        rx.errorbar(-t_mean, r, yerr=dr, fmt="o", ms=5, color=INK, mec="white", mew=1.2,
                    elinewidth=1.5, capsize=0, label="ratio of the spectra above", zorder=3)
        rx.set_yscale("log")
        
        rx.set_xlabel(r"$t = (p_{target}-p_{recoil})^2$ [GeV$^2$]", fontsize=9, color=INK)
        style(ax, left=(j == 0)); style(rx, left=(j == 0))

    axes[0, 0].set_ylabel(r"$dN/dt$ [events / GeV$^2$]", fontsize=9, color=INK)
    axes[0, 0].legend(frameon=False, fontsize=8, loc="lower left", labelcolor=INK2)
    axes[1, 0].set_ylim(0.3, 10)
    axes[1, 0].set_yticks([0.5, 1, 2, 5, 10], ["0.5", "1", "2", "5", "10"])
    axes[1, 0].yaxis.set_minor_formatter(plt.NullFormatter())
    axes[1, 0].set_ylabel(r"$R$", fontsize=10, color=INK)
    axes[1, 0].legend(frameon=False, fontsize=7.5, loc="upper right", labelcolor=INK2)
    sub.suptitle(name, fontsize=11, color=INK)

# Same spectrum scale in both blocks, so they compare directly.
lo_y = min(ax.get_ylim()[0] for ax in dist_rows); hi_y = max(ax.get_ylim()[1] for ax in dist_rows)
for ax in dist_rows:
    ax.set_ylim(lo_y, hi_y)
# fig.savefig("../t_spectra_ratio_labels_vs_none.pdf")